**Install PostgreSQL and Python connector**

In [1]:
!apt-get update -qq
!apt-get install -y postgresql postgresql-contrib -qq
!pip install -q psycopg2-binary

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Preconfiguring packages ...
Selecting previously unselected package libjson-perl.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../00-libjson-perl_4.10000-1_all.deb ...
Unpacking libjson-perl (4.10000-1) ...
Selecting previously unselected package postgresql-client-common.
Preparing to unpack .../01-postgresql-client-common_257build1.1_all.deb ...
Unpacking postgresql-client-common (257build1.1) ...
Selecting previously unselected package ssl-cert.
Preparing to unpack .../02-ssl-cert_1.1.2ubuntu1_all.deb ...
Unpacking ssl-cert (1.1.2ubuntu1) ...
Selecting previously unselected package postgresql-common.
Preparing to unpack .../03-postgresql-common_257build1.1_all.deb ...
Adding 'diversion of /usr/bin/pg_config to /usr/bin/pg_config.libpq-dev by postgres

In [8]:
!service postgresql start

[ OK ]


In [9]:
!service postgresql status

16/main (port 5432): online


**PostgreSQL database creation**

In [10]:
!sudo -u postgres psql -c "CREATE DATABASE task_manager;"

ERROR:  database "task_manager" already exists


In [11]:
!sudo -u postgres psql -d task_manager -f scripts/schema.sql

psql: error: scripts/schema.sql: No such file or directory


In [13]:
!find /content -name "schema.sql" 2>/dev/null

In [14]:
!ls

Day_4_CLI_Task_Manager.zip  sample_data


In [15]:
!unzip -o Day_4_CLI_Task_Manager.zip

Archive:  Day_4_CLI_Task_Manager.zip
   creating: day4_cli_task_manager/
   creating: day4_cli_task_manager/logs/
 extracting: day4_cli_task_manager/logs/.gitkeep  
   creating: day4_cli_task_manager/scripts/
  inflating: day4_cli_task_manager/scripts/schema.sql  
   creating: day4_cli_task_manager/tests/
  inflating: day4_cli_task_manager/tests/test_task_manager.py  
 extracting: day4_cli_task_manager/tests/__init__.py  
   creating: day4_cli_task_manager/exports/
 extracting: day4_cli_task_manager/exports/.gitkeep  
 extracting: day4_cli_task_manager/requirements.txt  
  inflating: day4_cli_task_manager/README.md  
  inflating: day4_cli_task_manager/.gitignore  
   creating: day4_cli_task_manager/task_manager/
  inflating: day4_cli_task_manager/task_manager/database.py  
  inflating: day4_cli_task_manager/task_manager/config.py  
  inflating: day4_cli_task_manager/task_manager/models.py  
  inflating: day4_cli_task_manager/task_manager/exporter.py  
  inflating: day4_cli_task_manager

In [16]:
!find /content -name "schema.sql" 2>/dev/null

/content/day4_cli_task_manager/scripts/schema.sql


In [17]:
%cd /content/day4_cli_task_manager

/content/day4_cli_task_manager


In [18]:
!ls

exports  logs  README.md  requirements.txt  scripts  task_manager  tests


In [19]:
!ls scripts

schema.sql


In [20]:
!sudo -u postgres psql -d task_manager -f scripts/schema.sql

CREATE TABLE
CREATE TABLE
INSERT 0 3
CREATE INDEX


In [26]:
!sudo -u postgres psql -d task_manager -c "\dt"

             List of relations
 Schema |     Name      | Type  |  Owner   
--------+---------------+-------+----------
 public | task_statuses | table | postgres
 public | tasks         | table | postgres
(2 rows)

>8

In [27]:
!sudo -u postgres psql -P pager=off -d task_manager -c "\dt"

             List of relations
 Schema |     Name      | Type  |  Owner   
--------+---------------+-------+----------
 public | task_statuses | table | postgres
 public | tasks         | table | postgres
(2 rows)



In [28]:
import os

os.environ["DB_HOST"] = "localhost"
os.environ["DB_PORT"] = "5432"
os.environ["DB_NAME"] = "task_manager"
os.environ["DB_USER"] = "postgres"
os.environ["DB_PASSWORD"] = ""

print("Database configuration set.")

Database configuration set.


In [29]:
!pip install -q psycopg2-binary

In [30]:
%cd /content/day4_cli_task_manager

/content/day4_cli_task_manager


In [36]:
!sudo -u postgres psql -c "ALTER USER postgres WITH PASSWORD 'postgres';"

ALTER ROLE


In [37]:
import os
os.environ["PGPASSWORD"] = "postgres"

!psql -h localhost -U postgres -d task_manager -c "SELECT current_user;"

 current_user 
--------------
 postgres
(1 row)



In [38]:
import os

os.environ["DB_HOST"] = "localhost"
os.environ["DB_PORT"] = "5432"
os.environ["DB_NAME"] = "task_manager"
os.environ["DB_USER"] = "postgres"
os.environ["DB_PASSWORD"] = "postgres"

In [39]:
import importlib
import task_manager.config

importlib.reload(task_manager.config)

print(task_manager.config.DB_CONFIG)

{'host': 'localhost', 'port': 5432, 'dbname': 'task_manager', 'user': 'postgres', 'password': 'postgres'}


**Direct PostgreSQL test**

In [40]:
!PGPASSWORD=postgres psql -h localhost -U postgres -d task_manager -c "SELECT current_user, current_database();"

 current_user | current_database 
--------------+------------------
 postgres     | task_manager
(1 row)



In [41]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    dbname="task_manager",
    user="postgres",
    password="postgres"
)

print("✅ psycopg2 connection successful!")

conn.close()

✅ psycopg2 connection successful!


In [42]:
import importlib
import task_manager.database

importlib.reload(task_manager.database)

from task_manager.database import Database

db = Database()
connection = db.connect()

print("✅ Project Database class connected successfully!")

connection.close()

✅ Project Database class connected successfully!


**test the actual database initialization**

In [43]:
db.initialize()

print("✅ Database initialization successful!")

✅ Database initialization successful!


In [44]:
!sudo -u postgres psql -P pager=off -d task_manager -c "\dt"

             List of relations
 Schema |     Name      | Type  |  Owner   
--------+---------------+-------+----------
 public | task_statuses | table | postgres
 public | tasks         | table | postgres
(2 rows)



In [45]:
from task_manager.models import Task

task = Task(
    title="Learn PostgreSQL",
    description="Practice CRUD operations",
    priority="high"
)

print(task)

Task(title='Learn PostgreSQL', description='Practice CRUD operations', status='pending', priority='high', id=None, created_at=None)


In [46]:
from task_manager.manager import TaskManager

manager = TaskManager(db)

print("TaskManager created successfully!")

TaskManager created successfully!


In [47]:
created_task = manager.add_task(task)

print("Created Task:")
print(created_task)

Created Task:
Task(title='Learn PostgreSQL', description='Practice CRUD operations', status='pending', priority='high', id=1, created_at=datetime.datetime(2026, 9, 21, 9, 26, 44, 463))


# **READ — retrieve the task**

In [48]:
tasks = manager.list_tasks()

print("All Tasks:")
for task in tasks:
    print(task)

All Tasks:
(1, 'Learn PostgreSQL', 'Practice CRUD operations', 'pending', 'high', datetime.datetime(2026, 9, 21, 9, 26, 44, 463))


In [49]:
pending_tasks = manager.list_tasks(status="pending")

print("Pending Tasks:")
for task in pending_tasks:
    print(task)

Pending Tasks:
(1, 'Learn PostgreSQL', 'Practice CRUD operations', 'pending', 'high', datetime.datetime(2026, 9, 21, 9, 26, 44, 463))


# **UPDATE**

change the task status from pending to in_progress

In [50]:
updated_task = manager.update_task(
    created_task.id,
    status="in_progress"
)

print("Updated Task:")
print(updated_task)

Updated Task:
(1, 'Learn PostgreSQL', 'Practice CRUD operations', 'in_progress', 'high', datetime.datetime(2026, 9, 21, 9, 26, 44, 463))


In [51]:
tasks = manager.list_tasks()

for task in tasks:
    print(task)

(1, 'Learn PostgreSQL', 'Practice CRUD operations', 'in_progress', 'high', datetime.datetime(2026, 9, 21, 9, 26, 44, 463))


# **UPDATE priority too**

 change the priority to medium

In [52]:
updated_task = manager.update_task(
    created_task.id,
    priority="medium"
)

print(updated_task)

(1, 'Learn PostgreSQL', 'Practice CRUD operations', 'in_progress', 'medium', datetime.datetime(2026, 9, 21, 9, 26, 44, 463))


# **DELETE**

In [53]:
deleted = manager.delete_task(created_task.id)

print("Task deleted:", deleted)

Task deleted: True


# Verify DELETE

In [54]:
tasks = manager.list_tasks()

print("Remaining tasks:")

for task in tasks:
    print(task)

Remaining tasks:


# **Simple Implementation of SQL**

In [55]:
task1 = Task(
    title="Learn SQL",
    description="Practice SQL queries",
    status="pending",
    priority="high"
)

task2 = Task(
    title="Learn Git",
    description="Practice Git commands",
    status="in_progress",
    priority="medium"
)

task3 = Task(
    title="Build Project",
    description="Complete Task Manager",
    status="completed",
    priority="high"
)

manager.add_task(task1)
manager.add_task(task2)
manager.add_task(task3)

print("3 practice tasks added.")

3 practice tasks added.


# **Aggregate Functions**

 Aggregate functions calculate a result from multiple rows.

In [56]:
import psycopg2

connection = db.connect()

with connection.cursor() as cursor:
    cursor.execute("SELECT COUNT(*) FROM tasks")
    result = cursor.fetchone()

print("Total tasks:", result[0])

connection.close()

Total tasks: 3


# **GROUP BY**

 GROUP BY groups rows having the same value.

In [57]:
connection = db.connect()

with connection.cursor() as cursor:
    cursor.execute("""
        SELECT status, COUNT(*)
        FROM tasks
        GROUP BY status
        ORDER BY status
    """)

    results = cursor.fetchall()

for row in results:
    print(row)

connection.close()

('completed', 1)
('in_progress', 1)
('pending', 1)


# **HAVING**

HAVING filters groups created by GROUP BY.

In [58]:
connection = db.connect()

with connection.cursor() as cursor:
    cursor.execute("""
        SELECT status, COUNT(*)
        FROM tasks
        GROUP BY status
        HAVING COUNT(*) > 0
    """)

    results = cursor.fetchall()

for row in results:
    print(row)

connection.close()

('completed', 1)
('pending', 1)
('in_progress', 1)


# **JOIN**
### A JOIN combines related data from multiple tables.

In [59]:
connection = db.connect()

with connection.cursor() as cursor:
    cursor.execute("""
        SELECT
            tasks.id,
            tasks.title,
            task_statuses.label
        FROM tasks
        JOIN task_statuses
            ON tasks.status = task_statuses.status
        ORDER BY tasks.id
    """)

    results = cursor.fetchall()

for row in results:
    print(row)

connection.close()

(2, 'Learn SQL', 'Pending')
(3, 'Learn Git', 'In Progress')
(4, 'Build Project', 'Completed')


# **Index**

schemas.sql

In [61]:
!sudo -u postgres psql -P pager=off -d task_manager -c "\di"

                       List of relations
 Schema |        Name        | Type  |  Owner   |     Table     
--------+--------------------+-------+----------+---------------
 public | idx_tasks_status   | index | postgres | tasks
 public | task_statuses_pkey | index | postgres | task_statuses
 public | tasks_pkey         | index | postgres | tasks
(3 rows)



# Test the SQL through  TaskManager

**GROUP BY + HAVING + aggregate**

In [74]:
stats = manager.statistics()

print(stats)

{'by_status': [('completed', 1), ('in_progress', 1), ('pending', 1)], 'totals': (3, 1)}


# **JOIN**
A join operation lets you retrieve data from two or more tables based on matching column values. The data in the tables is linked into a single result.

In [63]:
joined_tasks = manager.joined_tasks()

for task in joined_tasks:
    print(task)

(2, 'Learn SQL', 'Pending')
(3, 'Learn Git', 'In Progress')
(4, 'Build Project', 'Completed')


# **JSON Export**

test it with the tasks currently in PostgreSQL.

In [64]:
from task_manager.exporter import export_tasks

path = export_tasks(manager.list_tasks())

print("JSON exported to:", path)

JSON exported to: exports/tasks.json


In [66]:
!cat exports/tasks.json

[
    {
        "id": 2,
        "title": "Learn SQL",
        "description": "Practice SQL queries",
        "status": "pending",
        "priority": "high",
        "created_at": "2026-09-21 09:30:03.643906"
    },
    {
        "id": 3,
        "title": "Learn Git",
        "description": "Practice Git commands",
        "status": "in_progress",
        "priority": "medium",
        "created_at": "2026-09-21 09:30:03.668759"
    },
    {
        "id": 4,
        "title": "Build Project",
        "description": "Complete Task Manager",
        "status": "completed",
        "priority": "high",
        "created_at": "2026-09-21 09:30:03.699914"
    }
]

# **Logging**

 CLI uses Python's logging module.

Let's create a simple log entry:

In [67]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logging.info("Day 4 Task Manager test started")
logging.info("PostgreSQL CRUD operations completed")

In [68]:
!ls -la logs

total 8
drwxr-xr-x 2 root root 4096 Sep 21 05:43 .
drwxr-xr-x 7 root root 4096 Sep 21 05:43 ..
-rw-r--r-- 1 root root    0 Sep 21 05:43 .gitkeep


# **Runtime Error Handling**

 TaskManager validates task status and priority.

Let's intentionally test invalid data:

In [69]:
try:
    invalid_task = Task(
        title="Invalid Task",
        status="wrong_status"
    )

    manager.add_task(invalid_task)

except ValueError as error:
    print("Handled error:", error)

Handled error: Invalid status


In [70]:
try:
    invalid_task = Task(title="")
    manager.add_task(invalid_task)

except ValueError as error:
    print("Handled error:", error)

Handled error: Task title cannot be empty


# **Run the Tests**

Now let's test the project's unittest suite.

In [71]:
!python -m unittest discover -s tests -v

test_add (test_task_manager.TestTaskManager.test_add) ... ok
test_bad_status (test_task_manager.TestTaskManager.test_bad_status) ... ok
test_delete (test_task_manager.TestTaskManager.test_delete) ... ok
test_empty_title (test_task_manager.TestTaskManager.test_empty_title) ... ok
test_update (test_task_manager.TestTaskManager.test_update) ... ok

----------------------------------------------------------------------
Ran 5 tests in 0.003s

OK


# **Run the Actual CLI**



Let's run the actual application:

In [73]:
!python -m task_manager.cli

2026-09-21 09:37:50,137 | INFO | Connected to PostgreSQL

1.Add  2.List  3.Update  4.Delete  5.Search  6.Export JSON  7.Statistics  8.Exit
Choose: 1
Title: Politics
Description: Regional and National
Priority (low/medium/high): medium
2026-09-21 09:38:37,026 | INFO | Connected to PostgreSQL
Created: 5

1.Add  2.List  3.Update  4.Delete  5.Search  6.Export JSON  7.Statistics  8.Exit
Choose: 2
2026-09-21 09:38:43,236 | INFO | Connected to PostgreSQL
(2, 'Learn SQL', 'Practice SQL queries', 'pending', 'high', datetime.datetime(2026, 9, 21, 9, 30, 3, 643906))
(3, 'Learn Git', 'Practice Git commands', 'in_progress', 'medium', datetime.datetime(2026, 9, 21, 9, 30, 3, 668759))
(4, 'Build Project', 'Complete Task Manager', 'completed', 'high', datetime.datetime(2026, 9, 21, 9, 30, 3, 699914))
(5, 'Politics', 'Regional and National', 'pending', 'medium', datetime.datetime(2026, 9, 21, 9, 38, 37, 27257))

1.Add  2.List  3.Update  4.Delete  5.Search  6.Export JSON  7.Statistics  8.Exit
Choose: 4


In [76]:
%cd /content/day4_cli_task_manager

/content/day4_cli_task_manager


In [77]:
!pwd
!ls

/content/day4_cli_task_manager
exports  logs  README.md  requirements.txt  scripts  task_manager  tests


In [78]:
!git init

Reinitialized existing Git repository in /content/day4_cli_task_manager/.git/


In [79]:
!git config user.name "abhinandanCS05"
!git config user.email "abhinandanC5-G7@coastalseven.com"

In [80]:
!git config --list | grep user

user.name=abhinandanCS05
user.email=abhinandanC5-G7@coastalseven.com


In [81]:
!git status

On branch master

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.gitignore
	README.md
	exports/
	logs/
	requirements.txt
	scripts/
	task_manager/
	tests/

nothing added to commit but untracked files present (use "git add" to track)


In [82]:
!git add .

In [83]:
!git status

On branch master

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   .gitignore
	new file:   README.md
	new file:   exports/.gitkeep
	new file:   logs/.gitkeep
	new file:   requirements.txt
	new file:   scripts/schema.sql
	new file:   task_manager/__init__.py
	new file:   task_manager/cli.py
	new file:   task_manager/config.py
	new file:   task_manager/database.py
	new file:   task_manager/exporter.py
	new file:   task_manager/manager.py
	new file:   task_manager/models.py
	new file:   tests/__init__.py
	new file:   tests/test_task_manager.py



In [84]:
!git checkout -b day4-task-manager

Switched to a new branch 'day4-task-manager'


In [85]:
!git add .

In [86]:
!git commit -m "Initial Day 4 Task Manager"

[day4-task-manager (root-commit) dfc057e] Initial Day 4 Task Manager
 15 files changed, 313 insertions(+)
 create mode 100644 .gitignore
 create mode 100644 README.md
 create mode 100644 exports/.gitkeep
 create mode 100644 logs/.gitkeep
 create mode 100644 requirements.txt
 create mode 100644 scripts/schema.sql
 create mode 100644 task_manager/__init__.py
 create mode 100644 task_manager/cli.py
 create mode 100644 task_manager/config.py
 create mode 100644 task_manager/database.py
 create mode 100644 task_manager/exporter.py
 create mode 100644 task_manager/manager.py
 create mode 100644 task_manager/models.py
 create mode 100644 tests/__init__.py
 create mode 100644 tests/test_task_manager.py


In [87]:
!git status

On branch day4-task-manager
nothing to commit, working tree clean


In [88]:
!git remote -v

In [89]:
!git remote add origin https://github.com/abhinandanCS05/Coastal7_Learning_Internship.git

In [90]:
!git remote -v

origin	https://github.com/abhinandanCS05/Coastal7_Learning_Internship.git (fetch)
origin	https://github.com/abhinandanCS05/Coastal7_Learning_Internship.git (push)


In [91]:
!git ls-remote --heads origin

3e04948bf09d61354d6c9fb928a2aa5d1660b22d	refs/heads/Day-1-Python-Full-Stack-Self-Learning
67bf556660ea5fe0e536bfe971042fd5bc1cc915	refs/heads/Day-2-functions-and-data-structures
d1e7b2b1194ed0331bdf27ecbbb8838388dc2b3c	refs/heads/Day-3-OOPs-In-Python
ce9292523bb5e5d4635ce55612d9e029992009b3	refs/heads/main


In [92]:
!git push -u origin day4-task-manager

fatal: could not read Username for 'https://github.com': No such device or address


In [96]:
import subprocess
import os

result = subprocess.run(
    ["git", "push", "-u", "origin", "day4-task-manager"],
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result.stdout)

print("\nSTDERR:")
print(result.stderr)

print("\nExit code:", result.returncode)

STDOUT:


STDERR:
remote: Permission to abhinandanCS05/Coastal7_Learning_Internship.git denied to abhinandanCS05.
fatal: unable to access 'https://github.com/abhinandanCS05/Coastal7_Learning_Internship.git/': The requested URL returned error: 403


Exit code: 128


In [97]:
from getpass import getpass
import subprocess
import os

token = getpass("Enter your NEW GitHub token: ")

os.environ["GIT_USERNAME"] = "abhinandanCS05"
os.environ["GIT_TOKEN"] = token
os.environ["GIT_ASKPASS"] = "/tmp/git-askpass.sh"

with open("/tmp/git-askpass.sh", "w") as f:
    f.write("""#!/bin/sh
case "$1" in
  *Username*) printf '%s' "$GIT_USERNAME" ;;
  *Password*) printf '%s' "$GIT_TOKEN" ;;
esac
""")

os.chmod("/tmp/git-askpass.sh", 0o700)

result = subprocess.run(
    ["git", "push", "-u", "origin", "day4-task-manager"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

# Remove credential from environment and temporary script
os.environ.pop("GIT_TOKEN", None)
os.environ.pop("GIT_USERNAME", None)
os.environ.pop("GIT_ASKPASS", None)
os.remove("/tmp/git-askpass.sh")

Enter your NEW GitHub token: ··········
branch 'day4-task-manager' set up to track 'origin/day4-task-manager'.

remote: 
remote: Create a pull request for 'day4-task-manager' on GitHub by visiting:        
remote:      https://github.com/abhinandanCS05/Coastal7_Learning_Internship/pull/new/day4-task-manager        
remote: 
To https://github.com/abhinandanCS05/Coastal7_Learning_Internship.git
 * [new branch]      day4-task-manager -> day4-task-manager



In [98]:
!git branch -vv

* day4-task-manager dfc057e [origin/day4-task-manager] Initial Day 4 Task Manager


In [99]:
!git status
!find /content/day4_cli_task_manager -maxdepth 2 -type f | sort

On branch day4-task-manager
Your branch is up to date with 'origin/day4-task-manager'.

nothing to commit, working tree clean
/content/day4_cli_task_manager/exports/.gitkeep
/content/day4_cli_task_manager/exports/tasks.json
/content/day4_cli_task_manager/.git/COMMIT_EDITMSG
/content/day4_cli_task_manager/.git/config
/content/day4_cli_task_manager/.git/description
/content/day4_cli_task_manager/.git/HEAD
/content/day4_cli_task_manager/.gitignore
/content/day4_cli_task_manager/.git/index
/content/day4_cli_task_manager/logs/.gitkeep
/content/day4_cli_task_manager/logs/task_manager.log
/content/day4_cli_task_manager/README.md
/content/day4_cli_task_manager/requirements.txt
/content/day4_cli_task_manager/scripts/schema.sql
/content/day4_cli_task_manager/task_manager/cli.py
/content/day4_cli_task_manager/task_manager/config.py
/content/day4_cli_task_manager/task_manager/database.py
/content/day4_cli_task_manager/task_manager/exporter.py
/content/day4_cli_task_manager/task_manager/__init__.py